# 06 - Model Evaluation

## Objective

This notebook evaluates the final machine learning model using the unseen test dataset.

Evaluation includes:

- Confusion Matrix
- Classification Report
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Precision-Recall Curve
- Feature Importance
- Error Analysis
- Recommendations

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc
)

In [ ]:
df = pd.read_csv("../data/processed/engineered_transactions.csv")

df.head()

In [ ]:
if "transaction_id" in df.columns:
    df.drop("transaction_id", axis=1, inplace=True)

X = df.drop("is_fraud", axis=1)

y = df["is_fraud"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [ ]:
model = joblib.load("../models/fraud_model.pkl")

In [ ]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
accuracy = accuracy_score(y_test,y_pred)

precision = precision_score(y_test,y_pred)

recall = recall_score(y_test,y_pred)

f1 = f1_score(y_test,y_pred)

roc = roc_auc_score(y_test,y_prob)

print("Accuracy :",accuracy)

print("Precision:",precision)

print("Recall   :",recall)

print("F1 Score :",f1)

print("ROC AUC  :",roc)

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
cm = confusion_matrix(y_test,y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues"

)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")

plt.savefig("../reports/figures/confusion_matrix.png")

plt.close()

In [ ]:
fpr,tpr,threshold = roc_curve(y_test,y_prob)

plt.figure(figsize=(7,6))

plt.plot(

    fpr,

    tpr,

    label=f"ROC AUC = {roc:.3f}"

)

plt.plot([0,1],[0,1],"--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(7,6))

plt.plot(fpr,tpr)

plt.plot([0,1],[0,1],"--")

plt.savefig("../reports/figures/roc_curve.png")

plt.close()

In [ ]:
precision_curve,recall_curve,_ = precision_recall_curve(

    y_test,

    y_prob

)

pr_auc = auc(recall_curve,precision_curve)

plt.figure(figsize=(7,6))

plt.plot(

    recall_curve,

    precision_curve,

    label=f"PR AUC={pr_auc:.3f}"

)

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("Precision-Recall Curve")

plt.legend()

plt.show()

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importance = model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({

    "Feature":feature_names,

    "Importance":importance

})

importance_df = importance_df.sort_values(

    by="Importance",

    ascending=False

)

importance_df.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(

    data=importance_df.head(10),

    x="Importance",

    y="Feature"

)

plt.title("Top 10 Important Features")

plt.show()

In [ ]:
importance_df.to_csv(

    "../models/feature_importance.csv",

    index=False

)

In [ ]:
errors = X_test.copy()

errors["Actual"] = y_test.values

errors["Predicted"] = y_pred

misclassified = errors[

    errors["Actual"] != errors["Predicted"]

]

print("Misclassified Transactions")

misclassified.head()

# Error Analysis

The model makes two primary errors:

### False Positives

Legitimate transactions incorrectly classified as fraudulent.

Impact

- Additional manual investigation
- Increased operational cost

---

### False Negatives

Fraudulent transactions classified as legitimate.

Impact

- Financial loss
- Chargebacks
- Customer dissatisfaction

The fraud detection system should prioritize reducing False Negatives while maintaining acceptable Precision.

# Model Limitations

- Dataset may not represent every fraud pattern.
- Fraud behaviour changes over time.
- Model should be retrained periodically.
- Human analysts should review flagged transactions.
- Predictions should not be used as the sole basis for blocking transactions.

# Recommendation

The Random Forest model achieved the best overall performance.

It is recommended because:

- High Recall
- Good Precision
- Strong F1 Score
- Excellent ROC-AUC

The model can assist fraud analysts by prioritizing suspicious transactions for manual review.

Regular monitoring and retraining are recommended to maintain performance.

# Final Checklist

✔ Accuracy calculated

✔ Precision calculated

✔ Recall calculated

✔ F1 Score calculated

✔ ROC-AUC calculated

✔ Classification Report generated

✔ Confusion Matrix created

✔ ROC Curve plotted

✔ Precision-Recall Curve plotted

✔ Feature Importance generated

✔ Error Analysis completed

✔ Recommendations documented

Model evaluation completed successfully.